In [26]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [10]:
load_dotenv()

True

In [19]:
from dotenv import dotenv_values

config = dotenv_values(".env")
print(config.keys())

odict_keys(['GROQ_API_KEY'])


In [21]:
from dotenv import load_dotenv
import os

load_dotenv(".env")

print(os.getenv("GROQ_API_KEY") is not None)

True


In [22]:
llm=ChatGroq(model="openai/gpt-oss-120b")

In [23]:
#create the state
class LLMState(TypedDict):
    question:str
    answer:str

In [24]:
def llm_qa(state:LLMState)->LLMState:
    #extract the question from the state
    question=state['question']
    #generate a prompt
    prompt=f'answer the following question {question}'
    #call the llm
    answer=llm.invoke(prompt)
    state['answer']=answer
    return state

In [28]:
#create a graph 
graph=StateGraph(LLMState)

#add the nodes
graph.add_node('llm_qa',llm_qa)
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)
workflow=graph.compile()


In [30]:
initial_state={"question":"how far is moon from earth?"}
final_output=workflow.invoke(initial_state)
print(final_output['answer'])

content='The Moon’s distance from Earth isn’t a single fixed number because its orbit is elliptical, but the commonly quoted **average distance** is about **384\u202f400\u202fkilometers (≈238\u202f900\u202fmiles)**.\n\n- **Perigee (closest approach):** ~363\u202f300\u202fkm (≈225\u202f600\u202fmi)  \n- **Apogee (farthest point):** ~405\u202f500\u202fkm (≈252\u202f000\u202fmi)\n\nThese values can vary slightly from orbit to orbit due to gravitational influences from the Sun, Earth’s shape, and other factors. If you need the distance for a specific date or event, astronomers use precise ephemeris data (e.g., JPL Horizons) to compute the exact Earth‑Moon separation at that moment.' additional_kwargs={'reasoning_content': 'The user asks: "answer the following question how far is moon from earth?" We need to answer. Provide distance. Average distance about 384,400 km. Could mention range (perigee ~363,300 km, apogee ~405,500 km). Provide context. No policy issues. Just answer.'} response_me